In [14]:
# 1. 首先将10-14的气象30分钟数据合并成一张表，第二行单位需要去掉
from pathlib import Path
import pandas as pd
from typing import Tuple
import numpy as np

meteo_dir = Path("西双版纳橡胶林气象数据集")
years = range(2010, 2015)
frames = []
for year in years:
    file_path = meteo_dir / f"{year}年西双版纳橡胶林气象30分钟数据.xlsx"
    df_year = pd.read_excel(file_path)
    # 去掉第二行的单位说明行
    df_year = df_year.iloc[1:].reset_index(drop=True)
    df_year["来源年份"] = year
    frames.append(df_year)

df_meteo_30min = pd.concat(frames, ignore_index=True)
display(df_meteo_30min.head())

,年,月,日,时,分,秒,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,...,光合有效辐射,一层土壤温度,二层土壤温度,三层土壤温度,四层土壤温度,一层土壤体积含水量,二层土壤体积含水量,三层土壤体积含水量,降水量,来源年份
0,2010,6,24,19,30,0,29.42,29.39,66.28,64.01,...,20.27,27.11,26.39,26,24.88,NaN,NaN,NaN,0,2010
1,2010,6,24,20,0,0,28.06,28.13,75.68,70.94,...,7.013,26.92,26.39,26.03,24.88,0.277,0.285,0.184,0,2010
2,2010,6,24,20,30,0,26.68,27.02,84.4,78.13,...,0.478,26.64,26.38,26.05,24.88,0.286,0.294,0.19,0,2010
3,2010,6,24,21,0,0,25.74,26.32,91.5,81.4,...,0,26.42,26.35,26.07,24.87,0.286,0.294,0.19,0.2,2010
4,2010,6,24,21,30,0,25.28,26.09,93.5,83.9,...,0,26.21,26.33,26.1,24.88,0.286,0.294,0.19,0,2010


In [15]:
# 2. 将10-15年的通量30分钟数据合并成一张表，去除第二行单位信息
flux_dir = Path("西双版纳橡胶林通量数据集")
years_flux = range(2010, 2016)
flux_frames = []
missing_flux_files = []

for year in years_flux:
    flux_path = flux_dir / f"{year}年西双版纳橡胶林通量30分钟数据.xlsx"
    if not flux_path.exists():
        missing_flux_files.append(flux_path.name)
        continue
    df_flux_year = pd.read_excel(flux_path)
    df_flux_year = df_flux_year.iloc[1:].reset_index(drop=True)  # drop unit row
    df_flux_year["来源年份"] = year
    flux_frames.append(df_flux_year)

if not flux_frames:
    raise FileNotFoundError("未找到通量30分钟数据文件，请检查目录和文件名。")

df_flux_30min = pd.concat(flux_frames, ignore_index=True)
print("缺失文件:", missing_flux_files) if missing_flux_files else print("所有文件已加载")
display(df_flux_30min.head())

缺失文件: ['2015年西双版纳橡胶林通量30分钟数据.xlsx']


,年,月,日,时,分,NEE,RE,GPP,LE,Hs,ET,来源年份
0,2010,6,25,16,30,-7.0597,8.4161,15.476,246,-6.7998,0.101364,2010
1,2010,6,25,17,0,-6.324,8.5276,14.852,196,-6.4583,0.080799,2010
2,2010,6,25,17,30,-4.9257,8.4974,13.423,177,-21.2,0.072957,2010
3,2010,6,25,18,0,-5.6437,8.5903,14.234,193,23.4,0.079584,2010
4,2010,6,25,18,30,-1.6699,8.5299,10.2,107,-2.2658,0.04411,2010


In [16]:
# 3. 高效按日期对齐气象与通量数据（兼容“年/月/日/时/分”列）


def _find_first_column(df: pd.DataFrame, candidates) -> str | None:
    lower_map = {c.strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    return None


def normalize_datetime(df: pd.DataFrame, label: str) -> Tuple[pd.DataFrame, str]:
    df = df.copy()
    date_col = _find_first_column(df, ["date", "日期", "年月日", "日期(年月日)", "日期(年/月/日)"])
    time_col = _find_first_column(df, ["time", "时间", "时刻", "时间(时:分)", "时间(时/分)"])
    datetime_col = _find_first_column(
        df,
        ["datetime", "date_time", "date/time", "timestamp", "time_stamp", "时间戳", "日期时间", "时间戳（北京时间）", "时间戳(北京时间)"]
    )
    year_col = _find_first_column(df, ["year", "年", "年份"])
    month_col = _find_first_column(df, ["month", "月"])
    day_col = _find_first_column(df, ["day", "日", "天", "日子"])
    hour_col = _find_first_column(df, ["hour", "时", "小时"])
    minute_col = _find_first_column(df, ["minute", "分", "分钟"])
    second_col = _find_first_column(df, ["second", "秒", "秒钟"])

    used_cols = []
    if date_col and time_col:
        df["datetime"] = pd.to_datetime(
            df[date_col].astype(str).str.strip() + " " + df[time_col].astype(str).str.strip(), errors="coerce"
        )
        used_cols.extend([date_col, time_col])
    elif year_col and month_col and day_col:
        hour_series = df[hour_col] if hour_col else 0
        minute_series = df[minute_col] if minute_col else 0
        second_series = df[second_col] if second_col else 0
        df["datetime"] = pd.to_datetime(
            {
                "year": pd.to_numeric(df[year_col], errors="coerce"),
                "month": pd.to_numeric(df[month_col], errors="coerce"),
                "day": pd.to_numeric(df[day_col], errors="coerce"),
                "hour": pd.to_numeric(hour_series, errors="coerce"),
                "minute": pd.to_numeric(minute_series, errors="coerce"),
                "second": pd.to_numeric(second_series, errors="coerce"),
            }
        )
        used_cols.extend([c for c in [year_col, month_col, day_col, hour_col, minute_col, second_col] if c])
    elif datetime_col:
        df["datetime"] = pd.to_datetime(df[datetime_col], errors="coerce")
        used_cols.append(datetime_col)
    else:
        raise ValueError("未找到可用的日期/时间列，请检查原始数据列名。")

    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)
    for col in used_cols:
        if col in df.columns and col != "datetime":
            df = df.drop(columns=col)

    if "来源年份" in df.columns:
        df = df.rename(columns={"来源年份": f"来源年份_{label}"})

    return df, used_cols[0] if used_cols else ""


meteo_clean, _ = normalize_datetime(df_meteo_30min, "气象")
flux_clean, _ = normalize_datetime(df_flux_30min, "通量")

aligned = pd.merge(meteo_clean, flux_clean, on="datetime", how="inner", suffixes=("_meteo", "_flux"))
print(f"气象记录数: {len(meteo_clean)}, 通量记录数: {len(flux_clean)}, 对齐后: {len(aligned)}")
print("时间范围:", aligned["datetime"].min(), "→", aligned["datetime"].max())
display(aligned.head())

气象记录数: 79258, 通量记录数: 79216, 对齐后: 79216
时间范围: 2010-06-25 16:30:00 → 2015-01-01 00:00:00


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,...,降水量,来源年份_气象,datetime,NEE,RE,GPP,LE,Hs,ET,来源年份_通量
0,30.68,30.89,57.84,52.98,2.632,2.432,0.684,4.559,2.6147,921,...,0,2010,2010-06-25 16:30:00,-7.0597,8.4161,15.476,246,-6.7998,0.101364,2010
1,31.1,31.37,58.27,53.1,2.633,2.436,0.646,4.139,7.127129,934,...,0,2010,2010-06-25 17:00:00,-6.324,8.5276,14.852,196,-6.4583,0.080799,2010
2,30.81,31.24,59.32,53.59,2.636,2.44,0.729,4.225,21.75563,934,...,0,2010,2010-06-25 17:30:00,-4.9257,8.4974,13.423,177,-21.2,0.072957,2010
3,31.43,31.64,57.03,52.59,2.626,2.45,0.838,3.353,32.40912,934,...,0,2010,2010-06-25 18:00:00,-5.6437,8.5903,14.234,193,23.4,0.079584,2010
4,31.31,31.38,56.39,53.16,2.577,2.44,0.901,3.696,42.84151,934,...,0,2010,2010-06-25 18:30:00,-1.6699,8.5299,10.2,107,-2.2658,0.04411,2010


In [17]:
# 4. 筛选保留的字段并清除无关指标
columns_to_exclude = {"来源年份_气象", "来源年份_通量", "RE", "GPP", "LE", "Hs", "ET", "H"}
existing_excludes = [c for c in aligned.columns if c in columns_to_exclude]
print("实际剔除列:", existing_excludes)

aligned_filtered = aligned.drop(columns=existing_excludes)
print(f"过滤后列数: {aligned_filtered.shape[1]}")
display(aligned_filtered.head())

实际剔除列: ['来源年份_气象', 'RE', 'GPP', 'LE', 'Hs', 'ET', '来源年份_通量']
过滤后列数: 23


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,...,一层土壤温度,二层土壤温度,三层土壤温度,四层土壤温度,一层土壤体积含水量,二层土壤体积含水量,三层土壤体积含水量,降水量,datetime,NEE
0,30.68,30.89,57.84,52.98,2.632,2.432,0.684,4.559,2.6147,921,...,27.76,26.3,26.11,24.91,0.277,0.285,0.186,0,2010-06-25 16:30:00,-7.0597
1,31.1,31.37,58.27,53.1,2.633,2.436,0.646,4.139,7.127129,934,...,27.8,26.34,26.13,24.91,0.28,0.289,0.188,0,2010-06-25 17:00:00,-6.324
2,30.81,31.24,59.32,53.59,2.636,2.44,0.729,4.225,21.75563,934,...,27.59,26.4,26.15,24.91,0.28,0.289,0.188,0,2010-06-25 17:30:00,-4.9257
3,31.43,31.64,57.03,52.59,2.626,2.45,0.838,3.353,32.40912,934,...,27.64,26.44,26.17,24.91,0.28,0.289,0.188,0,2010-06-25 18:00:00,-5.6437
4,31.31,31.38,56.39,53.16,2.577,2.44,0.901,3.696,42.84151,934,...,27.45,26.49,26.19,24.91,0.28,0.289,0.188,0,2010-06-25 18:30:00,-1.6699


In [18]:
# 5. 平均前三层土壤温度/含水量并删除原始列（重写版）
# 说明：自动适配合并后可能存在的 _meteo/_flux 后缀；
# 在缺列的情况下按可用列求均值；完成后删除原始列与多余温度层。


# 基础列名
temp_bases = ["一层土壤温度", "二层土壤温度", "三层土壤温度"]
extra_temp_bases = ["四层土壤温度", "五层土壤温度", "六层土壤温度", "七层土壤温度", "八层土壤温度"]
moisture_bases = ["一层土壤体积含水量", "二层土壤体积含水量", "三层土壤体积含水量"]

# 查找列（按基名尝试：原名、_meteo、_flux）
def find_cols(df: pd.DataFrame, base_names):
    resolved = []
    for name in base_names:
        for cand in (name, f"{name}_meteo", f"{name}_flux"):
            if cand in df.columns:
                resolved.append(cand)
                break
    return resolved

# 基名缺失提示
def missing_bases(df: pd.DataFrame, base_names):
    miss = []
    for name in base_names:
        if not any(c in df.columns for c in (name, f"{name}_meteo", f"{name}_flux")):
            miss.append(name)
    return miss

# 解析存在的列
temp_cols_resolved = find_cols(aligned_filtered, temp_bases)
moisture_cols_resolved = find_cols(aligned_filtered, moisture_bases)
extra_temp_cols_resolved = find_cols(aligned_filtered, extra_temp_bases)

# 提示缺失
temp_missing = missing_bases(aligned_filtered, temp_bases)
moisture_missing = missing_bases(aligned_filtered, moisture_bases)
if temp_missing:
    print("缺失的土壤温度基列:", temp_missing)
if moisture_missing:
    print("缺失的土壤体积含水量基列:", moisture_missing)

# 按行求均值（跳过不可解析为数值的值）
if temp_cols_resolved:
    aligned_filtered["土壤温度"] = pd.concat(
        [pd.to_numeric(aligned_filtered[c], errors="coerce") for c in temp_cols_resolved], axis=1
    ).mean(axis=1, skipna=True)
else:
    print("未找到任何温度列，跳过温度均值计算。")

if moisture_cols_resolved:
    aligned_filtered["土壤含水量"] = pd.concat(
        [pd.to_numeric(aligned_filtered[c], errors="coerce") for c in moisture_cols_resolved], axis=1
    ).mean(axis=1, skipna=True)
else:
    print("未找到任何含水量列，跳过含水量均值计算。")

# 删除原始列（包含多余温度层）
drop_cols = [*temp_cols_resolved, *moisture_cols_resolved, *extra_temp_cols_resolved]
drop_cols = [c for c in drop_cols if c in aligned_filtered.columns]
aligned_filtered = aligned_filtered.drop(columns=drop_cols)
print("已删除原始列数:", len(drop_cols))

# 预览新列
preview_cols = [c for c in aligned_filtered.columns if "土壤温度" in c or "土壤含水量" in c]
display(aligned_filtered[preview_cols].head())

已删除原始列数: 7


,土壤温度,土壤含水量
0,26.723333,0.249333
1,26.756667,0.252333
2,26.713333,0.252333
3,26.750000,0.252333
4,26.710000,0.252333


In [19]:
# 6. 导出清洗后的合并结果供查看

# 确保存在清洗后的 DataFrame
try:
    export_df = aligned_filtered.copy()
except NameError:
    raise RuntimeError("未找到变量 aligned_filtered，请先运行前面的清洗步骤。")

# 排序并将 datetime 放到第一列
if "datetime" in export_df.columns:
    export_df = export_df.sort_values("datetime")
    cols = ["datetime"] + [c for c in export_df.columns if c != "datetime"]
    export_df = export_df[cols]

# 输出目录与文件名（与笔记本同级目录下）
out_dir = Path("西双版纳输出")
out_dir.mkdir(parents=True, exist_ok=True)
xlsx_path = out_dir / "合并清洗_西双版纳_30分钟_带均值.xlsx"
csv_path = out_dir / "合并清洗_西双版纳_30分钟_带均值.csv"

# 优先导出 Excel，失败则回退到 CSV
# try:
#     export_df.to_excel(xlsx_path, index=False)
#     print("已导出Excel:", xlsx_path)
# except Exception as e:
#     print("写入Excel失败，改写CSV。错误：", e)
#     export_df.to_csv(csv_path, index=False)
#     print("已导出CSV:", csv_path)


In [20]:
# 7. 统计关键列的缺失值与 -9999 计数

# 选择待统计的 DataFrame：优先使用导出用的 export_df，其次使用 aligned_filtered
try:
    df = export_df.copy()
except NameError:
    try:
        df = aligned_filtered.copy()
    except NameError:
        raise RuntimeError("未找到可用的 DataFrame，请先运行前面的合并与清洗步骤。")

# 关键列基名（自动适配可能的 _meteo/_flux 后缀）
base_cols = [
    "datetime",
    "近地面空气温度", "冠层上方空气温度",
    "近地面空气湿度", "冠层上方空气湿度",
    "近地面水汽压", "冠层上方水汽压",
    "近地面风速", "冠层上方风速",
    "风向", "大气压", "太阳辐射", "净辐射", "光合有效辐射",
    "降水量", "NEE", "土壤温度", "土壤含水量"
]

# 解析实际列名：按 原名 -> _meteo -> _flux 的优先级
def resolve_col(df: pd.DataFrame, base: str) -> str | None:
    for cand in (base, f"{base}_meteo", f"{base}_flux"):
        if cand in df.columns:
            return cand
    return None

total = len(df)
rows = []
for base in base_cols:
    col = resolve_col(df, base)
    if col is None:
        rows.append({
            "列基名": base,
            "实际列": "不存在",
            "总行数": total,
            "缺失值计数": None,
            "缺失占比%": None,
            "-9999计数": None,
            "-9999占比%": None,
        })
        continue

    s = df[col]
    miss = s.isna().sum()

    # datetime 不统计 -9999
    if base == "datetime":
        neg = 0
    else:
        s_num = pd.to_numeric(s, errors="coerce")
        neg = int((s_num == -9999).sum())

    miss_pct = round(miss / total * 100, 4) if total else 0.0
    neg_pct = round(neg / total * 100, 4) if total else 0.0

    rows.append({
        "列基名": base,
        "实际列": col,
        "总行数": total,
        "缺失值计数": int(miss),
        "缺失占比%": miss_pct,
        "-9999计数": int(neg),
        "-9999占比%": neg_pct,
    })

stats_df = pd.DataFrame(rows)

# 友好展示：将 datetime 放首行、其他按缺失占比降序
order = ["datetime"] + [b for b in base_cols if b != "datetime"]
stats_df["_order"] = stats_df["列基名"].apply(lambda x: order.index(x) if x in order else len(order))
stats_df = stats_df.sort_values(["_order", "缺失占比%"], ascending=[True, False]).drop(columns=["_order"]).reset_index(drop=True)

display(stats_df)

# 导出统计结果到输出目录
# out_dir = Path("西双版纳输出")
# out_dir.mkdir(parents=True, exist_ok=True)
# stats_path = out_dir / "质量统计_缺失与-9999_西双版纳_30分钟.csv"
# stats_df.to_csv(stats_path, index=False)
# print("统计结果已导出:", stats_path)


,列基名,实际列,总行数,缺失值计数,缺失占比%,-9999计数,-9999占比%
0,datetime,datetime,79216,0,0.0000,0,0.0000
1,近地面空气温度,近地面空气温度,79216,13,0.0164,0,0.0000
2,冠层上方空气温度,冠层上方空气温度,79216,1,0.0013,0,0.0000
3,近地面空气湿度,近地面空气湿度,79216,13,0.0164,0,0.0000
4,冠层上方空气湿度,冠层上方空气湿度,79216,0,0.0000,0,0.0000
5,近地面水汽压,近地面水汽压,79216,13,0.0164,0,0.0000
6,冠层上方水汽压,冠层上方水汽压,79216,13,0.0164,0,0.0000
7,近地面风速,近地面风速,79216,2608,3.2923,0,0.0000
8,冠层上方风速,冠层上方风速,79216,2608,3.2923,0,0.0000
9,风向,风向,79216,2608,3.2923,0,0.0000


In [21]:
# 8. 两两线性关系统计与导出（自动解析列后缀，忽略-9999）


# 选择用于分析的 DataFrame：优先使用 export_df，其次 aligned_filtered
try:
    df_src = export_df.copy()
except NameError:
    try:
        df_src = aligned_filtered.copy()
    except NameError:
        raise RuntimeError("未找到可用的 DataFrame，请先运行前面的合并与清洗步骤。")

# 基础列基名（与前面统计一致），剔除 datetime 做数值分析
base_cols = [
    "近地面空气温度", "冠层上方空气温度",
    "近地面空气湿度", "冠层上方空气湿度",
    "近地面水汽压", "冠层上方水汽压",
    "近地面风速", "冠层上方风速",
    "风向", "大气压", "太阳辐射", "净辐射", "光合有效辐射",
    "降水量", "NEE", "土壤温度", "土壤含水量"
]

# 按 原名 -> _meteo -> _flux 的优先级解析实际列名
def resolve_col(df: pd.DataFrame, base: str) -> str | None:
    for cand in (base, f"{base}_meteo", f"{base}_flux"):
        if cand in df.columns:
            return cand
    return None

resolved_map: dict[str, str] = {}
for b in base_cols:
    col = resolve_col(df_src, b)
    if col is not None:
        resolved_map[b] = col

if not resolved_map:
    raise RuntimeError("未解析到任何目标列，请检查上一节导出的列名是否匹配。")

# 构造清洗后的数值 DataFrame（将不可解析与 -9999 置为 NaN）
num_cols_actual = list(resolved_map.values())
clean_data = {}
for actual in num_cols_actual:
    s = pd.to_numeric(df_src[actual], errors="coerce")
    s = s.where(s != -9999, np.nan)
    clean_data[actual] = s

num_df = pd.DataFrame(clean_data)

# 计算 Pearson 相关矩阵（按实际列名）
corr = num_df.corr(method="pearson")

# 计算两两线性回归（y ~ slope * x + intercept）
rows = []
# 反向映射：实际列 -> 基名，便于展示
base_by_actual = {v: k for k, v in resolved_map.items()}

actuals = [c for c in num_df.columns]
for i in range(len(actuals)):
    for j in range(i + 1, len(actuals)):
        xi = actuals[i]
        yj = actuals[j]
        x = num_df[xi].to_numpy()
        y = num_df[yj].to_numpy()
        mask = (~np.isnan(x)) & (~np.isnan(y))
        n = int(mask.sum())
        if n >= 2:
            # 最小二乘一阶拟合
            try:
                slope, intercept = np.polyfit(x[mask], y[mask], 1)
            except Exception:
                slope, intercept = np.nan, np.nan
            # Pearson r
            try:
                r = float(np.corrcoef(x[mask], y[mask])[0, 1])
            except Exception:
                r = np.nan
            r2 = float(r * r) if not np.isnan(r) else np.nan
        else:
            slope, intercept, r, r2 = np.nan, np.nan, np.nan, np.nan

        rows.append({
            "变量X基名": base_by_actual.get(xi, xi),
            "变量X实际列": xi,
            "变量Y基名": base_by_actual.get(yj, yj),
            "变量Y实际列": yj,
            "样本数n": n,
            "Pearson_r": r,
            "R2": r2,
            "线性斜率": slope,
            "线性截距": intercept,
        })

pair_df = pd.DataFrame(rows)
pair_df = pair_df.sort_values(["R2", "样本数n"], ascending=[False, False]).reset_index(drop=True)


# 预览前几行
display(pair_df.head(20))
display(corr)

,变量X基名,变量X实际列,变量Y基名,变量Y实际列,样本数n,Pearson_r,R2,线性斜率,线性截距
0,近地面空气温度,近地面空气温度,冠层上方空气温度,冠层上方空气温度,79202,0.982212,0.964740,1.020679,-0.541469
1,太阳辐射,太阳辐射,光合有效辐射,光合有效辐射,79203,0.948419,0.899498,1.758846,12.506817
2,近地面空气湿度,近地面空气湿度,冠层上方空气湿度,冠层上方空气湿度,79203,0.937092,0.878141,0.915996,2.548857
3,太阳辐射,太阳辐射,净辐射,净辐射,64075,0.880351,0.775017,0.840074,-3.467513
4,净辐射,净辐射,光合有效辐射,光合有效辐射,64075,0.862847,0.744505,1.641757,66.501839
5,近地面风速,近地面风速,冠层上方风速,冠层上方风速,76608,0.726377,0.527624,2.313933,0.431656
6,近地面空气温度,近地面空气温度,土壤温度,土壤温度,79203,0.701730,0.492425,0.380627,14.361161
7,冠层上方空气温度,冠层上方空气温度,土壤温度,土壤温度,79202,0.700901,0.491262,0.365850,14.716587
8,冠层上方水汽压,冠层上方水汽压,土壤温度,土壤温度,79202,0.690503,0.476794,2.937854,16.115771
9,光合有效辐射,光合有效辐射,NEE,NEE,79203,-0.679606,0.461864,-0.017338,3.279701


,近地面空气温度,冠层上方空气温度,近地面空气湿度,冠层上方空气湿度,近地面水汽压,冠层上方水汽压,近地面风速,冠层上方风速,风向,大气压,太阳辐射,净辐射,光合有效辐射,降水量,NEE,土壤温度,土壤含水量
近地面空气温度,1.000000,0.982212,-0.543380,-0.568846,0.265605,0.333434,0.345856,0.392696,0.105021,-0.176379,0.450176,0.382324,0.471974,0.006755,-0.266242,0.701730,0.092686
冠层上方空气温度,0.982212,1.000000,-0.556049,-0.594554,0.261266,0.300679,0.333990,0.390910,0.118487,-0.179755,0.429490,0.366062,0.446247,0.002439,-0.255830,0.700901,0.074234
近地面空气湿度,-0.543380,-0.556049,1.000000,0.937092,0.193877,0.447159,-0.576344,-0.520225,-0.101176,0.045955,-0.577630,-0.508474,-0.608798,0.045454,0.313546,-0.016142,0.196620
冠层上方空气湿度,-0.568846,-0.594554,0.937092,1.000000,0.152522,0.354497,-0.542932,-0.508209,-0.090021,0.071257,-0.564843,-0.490610,-0.562883,0.035850,0.299739,-0.060741,0.164860
近地面水汽压,0.265605,0.261266,0.193877,0.152522,1.000000,0.441178,-0.094070,-0.028397,0.031584,-0.083984,-0.030132,-0.016183,-0.032781,0.029159,-0.008143,0.441169,0.185369
冠层上方水汽压,0.333434,0.300679,0.447159,0.354497,0.441178,1.000000,-0.230719,-0.159081,-0.014577,-0.130105,-0.122393,-0.073959,-0.121891,0.054097,0.047756,0.690503,0.347463
近地面风速,0.345856,0.333990,-0.576344,-0.542932,-0.094070,-0.230719,1.000000,0.726377,0.154838,-0.043451,0.498173,0.436269,0.502181,0.057315,-0.253528,0.025757,-0.129862
冠层上方风速,0.392696,0.390910,-0.520225,-0.508209,-0.028397,-0.159081,0.726377,1.000000,0.313407,-0.071480,0.308704,0.260621,0.306842,0.085789,-0.152936,0.149817,-0.051421
风向,0.105021,0.118487,-0.101176,-0.090021,0.031584,-0.014577,0.154838,0.313407,1.000000,-0.023327,0.068599,0.006628,0.069203,0.019373,-0.046047,0.114616,0.018427
大气压,-0.176379,-0.179755,0.045955,0.071257,-0.083984,-0.130105,-0.043451,-0.071480,-0.023327,1.000000,-0.010281,-0.008053,-0.009902,-0.010295,-0.007181,-0.195304,0.126677


In [22]:
# 9. 删除指定的协变量列

# 指定要删除的协变量（按基名；自动解析 _meteo/_flux 后缀）
columns_to_remove = [
    "近地面空气温度",
    "近地面空气湿度",
    "净辐射",
    "光合有效辐射",
    "降水量",
    "近地面水汽压",
    "大气压",
    "冠层上方风速",
]

# 选择数据来源：优先使用导出的 export_df，其次使用 aligned_filtered
try:
    df_src = export_df.copy()
except NameError:
    try:
        df_src = aligned_filtered.copy()
    except NameError:
        raise RuntimeError("未找到可用的 DataFrame，请先运行前面的合并与清洗步骤。")

# 解析实际列名（原名 -> _meteo -> _flux）
def resolve_col(df: pd.DataFrame, base: str) -> str | None:
    for cand in (base, f"{base}_meteo", f"{base}_flux"):
        if cand in df.columns:
            return cand
    return None

remove_actual = []
missing_bases = []
for base in columns_to_remove:
    col = resolve_col(df_src, base)
    if col is None:
        missing_bases.append(base)
    else:
        remove_actual.append(col)

# 执行删除并生成新的 DataFrame
reduced_df = df_src.drop(columns=remove_actual, errors="ignore")
print("计划删除基名:", columns_to_remove)
print("实际删除列:", remove_actual)
if missing_bases:
    print("未找到的基名（可能不存在或命名不同）:", missing_bases)
print(f"删除后列数: {reduced_df.shape[1]}，行数: {reduced_df.shape[0]}")

# 将结果保存为新的变量，便于后续建模使用
df_for_model = reduced_df.copy()

# 预览前几行
display(df_for_model.head())

计划删除基名: ['近地面空气温度', '近地面空气湿度', '净辐射', '光合有效辐射', '降水量', '近地面水汽压', '大气压', '冠层上方风速']
实际删除列: ['近地面空气温度', '近地面空气湿度', '净辐射', '光合有效辐射', '降水量', '近地面水汽压', '大气压', '冠层上方风速']
删除后列数: 10，行数: 79216


,datetime,冠层上方空气温度,冠层上方空气湿度,冠层上方水汽压,近地面风速,风向,太阳辐射,NEE,土壤温度,土壤含水量
0,2010-06-25 16:30:00,30.89,52.98,2.432,0.684,2.6147,697.1,-7.0597,26.723333,0.249333
1,2010-06-25 17:00:00,31.37,53.1,2.436,0.646,7.127129,267.6,-6.324,26.756667,0.252333
2,2010-06-25 17:30:00,31.24,53.59,2.44,0.729,21.75563,236.4,-4.9257,26.713333,0.252333
3,2010-06-25 18:00:00,31.64,52.59,2.45,0.838,32.40912,427.2,-5.6437,26.750000,0.252333
4,2010-06-25 18:30:00,31.38,53.16,2.44,0.901,42.84151,157.2,-1.6699,26.710000,0.252333


In [23]:
var_name_map = {
    "datetime": "date",

    "冠层上方空气温度": "Ta",      # Air temperature above canopy
    "冠层上方空气湿度": "RH",      # Relative humidity above canopy
    "冠层上方水汽压": "VPD",       # Vapor pressure (或水汽压) above canopy

    "近地面风速": "WS",           # Wind speed near surface
    "风向": "WD",                         # Wind direction
    "太阳辐射": "SW",                     # Shortwave radiation / Solar radiation
    "NEE": "NEE",                         # Net Ecosystem Exchange
    "土壤温度": "Ts",                     # Soil temperature
    "土壤含水量": "SWC"                   # Soil water content
}


In [24]:
# 9. 字段映射与列顺序调整（date 放首列，NEE 放末列；自动解析 _meteo/_flux）
# 基础 DataFrame：优先使用 export_df（已清洗导出），否则回退 aligned_filtered
try:
    df_src = export_df.copy()
except NameError:
    try:
        df_src = aligned_filtered.copy()
    except NameError:
        raise RuntimeError("未找到可用的 DataFrame，请先运行前面的清洗步骤。")

# 映射字典（源列名 -> 目标名）
var_name_map = {
    "datetime": "date",
    "冠层上方空气温度": "Ta",
    "冠层上方空气湿度": "RH",
    "冠层上方水汽压": "VPD",
    "近地面风速": "WS",
    "风向": "WD",
    "太阳辐射": "SW",
    "NEE": "NEE",
    "土壤温度": "Ts",
    "土壤含水量": "SWC",
}

# 解析列：按 原名 -> _meteo -> _flux 的优先级
def resolve_col(df: pd.DataFrame, base: str) -> str | None:
    for cand in (base, f"{base}_meteo", f"{base}_flux"):
        if cand in df.columns:
            return cand
    return None

resolved = {}
missing = []
for base, new_name in var_name_map.items():
    col = resolve_col(df_src, base)
    if col is None:
        missing.append(base)
    else:
        resolved[new_name] = col

if missing:
    print("未找到的源列（跳过映射）:", missing)

# 构建映射后的 DataFrame
out = pd.DataFrame()
for new_name, actual_col in resolved.items():
    s = df_src[actual_col]
    if new_name == "date":
        # 尝试标准化为时间类型
        s = pd.to_datetime(s, errors="coerce")
    else:
        s = pd.to_numeric(s, errors="coerce")
        s = s.where(s != -9999, np.nan)
    out[new_name] = s

# 列顺序：date 放第一列、NEE 放最后一列，其余保持当前顺序
cols = list(out.columns)
if "date" in cols:
    cols = ["date"] + [c for c in cols if c not in ("date", "NEE")] + (["NEE"] if "NEE" in cols else [])
elif "NEE" in cols:
    cols = [c for c in cols if c != "NEE"] + ["NEE"]
out = out[cols]

display(out.head())

,date,Ta,RH,VPD,WS,WD,SW,Ts,SWC,NEE
0,2010-06-25 16:30:00,30.89,52.98,2.432,0.684,2.614700,697.1,26.723333,0.249333,-7.0597
1,2010-06-25 17:00:00,31.37,53.10,2.436,0.646,7.127129,267.6,26.756667,0.252333,-6.3240
2,2010-06-25 17:30:00,31.24,53.59,2.440,0.729,21.755630,236.4,26.713333,0.252333,-4.9257
3,2010-06-25 18:00:00,31.64,52.59,2.450,0.838,32.409120,427.2,26.750000,0.252333,-5.6437
4,2010-06-25 18:30:00,31.38,53.16,2.440,0.901,42.841510,157.2,26.710000,0.252333,-1.6699


In [25]:
# 10. 清洗：去除包含空值或 -9999 的整行（基于映射结果）
# 优先使用上一步的映射结果 DataFrame：out
# 若不存在，则根据 var_name_map 与合并后的数据重建一次
try:
    mapped = out.copy()
    print("检测到已映射数据 out，直接使用。")
except NameError:
    print("未检测到 out，尝试根据 var_name_map 重建映射数据……")
    # 选择数据来源：优先 export_df，其次 aligned_filtered
    try:
        df_src = export_df.copy()
    except NameError:
        try:
            df_src = aligned_filtered.copy()
        except NameError:
            raise RuntimeError("未找到可用的 DataFrame，请先运行前面的步骤。")
    # 若 var_name_map 未定义，则内置一份默认映射
    try:
        var_name_map  # noqa: F401
    except NameError:
        var_name_map = {
            "datetime": "date",
            "冠层上方空气温度": "Ta",
            "冠层上方空气湿度": "RH",
            "冠层上方水汽压": "VPD",
            "近地面风速": "WS",
            "风向": "WD",
            "太阳辐射": "SW",
            "NEE": "NEE",
            "土壤温度": "Ts",
            "土壤含水量": "SWC",
        }
    # 解析列：按 原名 -> _meteo -> _flux 的优先级
    def resolve_col(df: pd.DataFrame, base: str) -> str | None:
        for cand in (base, f"{base}_meteo", f"{base}_flux"):
            if cand in df.columns:
                return cand
        return None
    resolved = {}
    missing = []
    for base, new_name in var_name_map.items():
        col = resolve_col(df_src, base)
        if col is None:
            missing.append(base)
        else:
            resolved[new_name] = col
    if missing:
        print("未找到的源列（跳过映射）:", missing)
    mapped = pd.DataFrame()
    for new_name, actual_col in resolved.items():
        s = df_src[actual_col]
        if new_name == "date":
            s = pd.to_datetime(s, errors="coerce")
        else:
            s = pd.to_numeric(s, errors="coerce")
            s = s.where(s != -9999, np.nan)
        mapped[new_name] = s

# 清洗逻辑：
# 1) 非 date 列将 -9999 统一视为缺失（NaN）
# 2) 丢弃任一列为 NaN 的整行
mapped_clean = mapped.copy()
num_cols = [c for c in mapped_clean.columns if c != "date"]
for c in num_cols:
    mapped_clean[c] = pd.to_numeric(mapped_clean[c], errors="coerce")
    mapped_clean[c] = mapped_clean[c].where(mapped_clean[c] != -9999, np.nan)

before_n = len(mapped_clean)
mapped_clean = mapped_clean.dropna(how="any")
after_n = len(mapped_clean)
dropped = before_n - after_n
print(f"清洗完成：原始 {before_n} 行 → 保留 {after_n} 行（删除 {dropped} 行）")

# 可选：按日期排序
if "date" in mapped_clean.columns:
    mapped_clean = mapped_clean.sort_values("date").reset_index(drop=True)

# 导出结果
out_dir = Path("西双版纳输出")
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "XiShuangBanNa.csv"

mapped_clean.to_csv(csv_path, index=False)
print("已导出CSV:", csv_path)

# 暴露清洗后的变量，便于后续使用
out_clean = mapped_clean
display(out_clean.head())

检测到已映射数据 out，直接使用。
清洗完成：原始 79216 行 → 保留 76606 行（删除 2610 行）


已导出CSV: 西双版纳输出/XiShuangBanNa.csv


,date,Ta,RH,VPD,WS,WD,SW,Ts,SWC,NEE
0,2010-06-25 16:30:00,30.89,52.98,2.432,0.684,2.614700,697.1,26.723333,0.249333,-7.0597
1,2010-06-25 17:00:00,31.37,53.10,2.436,0.646,7.127129,267.6,26.756667,0.252333,-6.3240
2,2010-06-25 17:30:00,31.24,53.59,2.440,0.729,21.755630,236.4,26.713333,0.252333,-4.9257
3,2010-06-25 18:00:00,31.64,52.59,2.450,0.838,32.409120,427.2,26.750000,0.252333,-5.6437
4,2010-06-25 18:30:00,31.38,53.16,2.440,0.901,42.841510,157.2,26.710000,0.252333,-1.6699


In [28]:
# # 11. 随机森林快速建模：以 NEE 为目标，特征为 Ta/RH/VPD/WS/WD/SW/Ts/SWC
# from pathlib import Path
# import numpy as np
# import pandas as pd
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# # 1) 取数据：优先使用上一节清洗结果 out_clean；若不存在，尝试读取导出的 CSV
# try:
#     df_model = out_clean.copy()
#     print("检测到内存中的 out_clean，直接使用。")
# except NameError:
#     out_path = Path("西双版纳输出") / "XiShuangBanNa.csv"
#     if out_path.exists():
#         df_model = pd.read_csv(out_path, parse_dates=["date"])
#         print("从磁盘读取已清洗数据:", out_path)
#     else:
#         raise RuntimeError("未找到 out_clean 或 XiShuangBanNa.csv，请先运行映射与清洗步骤。")

# # 2) 选择特征与标签
# feature_cols = ["Ta", "RH", "VPD", "WS", "WD", "SW", "Ts", "SWC"]
# target_col = "NEE"
# missing_feats = [c for c in feature_cols + [target_col] if c not in df_model.columns]
# if missing_feats:
#     raise RuntimeError(f"缺少以下必要列: {missing_feats}")

# # 保留所需列，确保无缺失
# work = df_model[["date"] + feature_cols + [target_col]].copy()
# # 安全：将 -9999 视为缺失再丢弃
# for c in feature_cols + [target_col]:
#     work[c] = pd.to_numeric(work[c], errors="coerce")
#     work[c] = work[c].where(work[c] != -9999, np.nan)
# before = len(work)
# work = work.dropna(how="any").reset_index(drop=True)
# after = len(work)
# print(f"清洗后样本数: {after}（删除 {before-after} 行含缺失/-9999 的记录）")

# # 3) 按时间划分训练/测试（最后 20% 作为测试集）
# if "date" in work.columns:
#     work = work.sort_values("date").reset_index(drop=True)
# n = len(work)
# split_idx = int(n * 0.8) if n > 10 else max(1, n - 1)
# train_df = work.iloc[:split_idx].copy()
# test_df = work.iloc[split_idx:].copy()

# X_train = train_df[feature_cols].to_numpy()
# y_train = train_df[target_col].to_numpy().ravel()
# X_test = test_df[feature_cols].to_numpy()
# y_test = test_df[target_col].to_numpy().ravel()

# print(f"训练样本: {X_train.shape[0]}，测试样本: {X_test.shape[0]}")

# # 4) 训练随机森林回归模型（快速基线）
# rf = RandomForestRegressor(
#     n_estimators=200,
#     max_depth=None,
#     random_state=42,
#     n_jobs=-1,
#     min_samples_split=2,
#     min_samples_leaf=1,
#     oob_score=False
# )
# rf.fit(X_train, y_train)

# # 5) 评估
# pred = rf.predict(X_test)
# mae = mean_absolute_error(y_test, pred)
# rmse = float(np.sqrt(mean_squared_error(y_test, pred)))
# r2 = r2_score(y_test, pred)
# print({"MAE": round(mae, 4), "RMSE": round(rmse, 4), "R2": round(r2, 4)})

# # 6) 特征重要性与结果导出
# imp = pd.DataFrame({
#     "feature": feature_cols,
#     "importance": rf.feature_importances_,
# }).sort_values("importance", ascending=False).reset_index(drop=True)

# result_df = pd.DataFrame({
#     "date": test_df["date"].to_numpy() if "date" in test_df.columns else np.arange(len(pred)),
#     "NEE_true": y_test,
#     "NEE_pred": pred,
#     "residual": y_test - pred,
# })

# out_dir = Path("西双版纳输出")
# out_dir.mkdir(parents=True, exist_ok=True)
# pred_path = out_dir / "RF_pred_西双版纳_NEE.csv"
# imp_path = out_dir / "RF_feature_importance_西双版纳_NEE.csv"
# result_df.to_csv(pred_path, index=False)
# imp.to_csv(imp_path, index=False)
# print("已导出预测结果:", pred_path)
# print("已导出特征重要性:", imp_path)

# # 7) 预览
# display(imp)
# display(result_df.head(20))

检测到内存中的 out_clean，直接使用。
清洗后样本数: 76606（删除 0 行含缺失/-9999 的记录）
训练样本: 61284，测试样本: 15322
{'MAE': 4.3013, 'RMSE': 6.0823, 'R2': 0.6751}
已导出预测结果: 西双版纳输出/RF_pred_西双版纳_NEE.csv
已导出特征重要性: 西双版纳输出/RF_feature_importance_西双版纳_NEE.csv


检测到内存中的 out_clean，直接使用。
清洗后样本数: 76606（删除 0 行含缺失/-9999 的记录）
训练样本: 61284，测试样本: 15322
{'MAE': 4.3013, 'RMSE': 6.0823, 'R2': 0.6751}
已导出预测结果: 西双版纳输出/RF_pred_西双版纳_NEE.csv
已导出特征重要性: 西双版纳输出/RF_feature_importance_西双版纳_NEE.csv


,feature,importance
0,SW,0.708275
1,RH,0.074479
2,Ts,0.045277
3,SWC,0.041117
4,Ta,0.034780
5,VPD,0.033630
6,WS,0.032452
7,WD,0.029989


,date,NEE_true,NEE_pred,residual
0,2014-02-12 15:00:00,-16.70800,-1.580562,-15.127438
1,2014-02-12 15:30:00,-21.97300,0.292699,-22.265699
2,2014-02-12 16:00:00,-2.89880,-0.519303,-2.379497
3,2014-02-12 16:30:00,-10.04100,0.596801,-10.637801
4,2014-02-12 17:00:00,-1.46000,0.966442,-2.426442
5,2014-02-12 17:30:00,-0.17691,2.401123,-2.578033
6,2014-02-12 18:00:00,2.98520,2.617921,0.367279
7,2014-02-12 18:30:00,-0.88911,3.461680,-4.350790
8,2014-02-12 19:00:00,-2.16450,5.301430,-7.465930
9,2014-02-12 19:30:00,0.25306,2.884868,-2.631808
